# Phishing Website Detection Using Machine Learning
**Prepared by:** Shubhangi Waghwase

## Project objective
Build and evaluate machine-learning classifiers that identify whether a website is phishing or legitimate using the UCI Phishing Websites dataset.

> **Important:** This is an educational model trained on a historical dataset. It is not a live website scanner and must not be used as the only basis for deciding whether a website is safe.

## Dataset
UCI Machine Learning Repository — [Phishing Websites (Dataset 327)](https://archive.ics.uci.edu/dataset/327/phishing+websites)

The dataset contains 11,055 records and 30 input features, with a target label. It includes URL and website characteristics such as IP-address use, URL length, shortening services, `@` symbols, redirects, domain information, and HTTPS-related signals.

## Workflow
1. Install/import dependencies.
2. Fetch and inspect the dataset.
3. Prepare features and labels.
4. Split into training and testing sets.
5. Train Decision Tree, Logistic Regression, and AdaBoost models.
6. Compare accuracy, precision, recall, F1-score, and confusion matrices.

In [ ]:
# Install dependencies if they are not already installed:
# %pip install -r requirements.txt

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42

## 1. Load the dataset

In [ ]:
# Fetch dataset 327 from the UCI Machine Learning Repository
dataset = fetch_ucirepo(id=327)

X = dataset.data.features.copy()
y = dataset.data.targets.copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
display(y.head())
print("\\nTarget columns:", list(y.columns))
print("\\nTarget values:")
print(y.iloc[:, 0].value_counts(dropna=False))

## 2. Prepare the data

The UCI dataset uses `-1` for phishing and `1` for legitimate websites. The code below normalizes the target to readable class names. It also handles missing values and converts categorical feature columns into numeric form when needed.

In [ ]:
# Use the first target column supplied by UCI
target = y.iloc[:, 0].copy()

# Normalize labels. For the UCI Phishing Websites dataset:
# -1 = phishing, 1 = legitimate.
def label_name(value):
    text = str(value).strip().lower()
    if text in {"-1", "-1.0", "phishing", "phishy"}:
        return "Phishing"
    if text in {"1", "1.0", "legitimate", "legit"}:
        return "Legitimate"
    if text in {"0", "0.0", "suspicious"}:
        return "Suspicious"
    return str(value)

y_labels = target.map(label_name)

# Convert all features to numeric values; non-numeric categories are one-hot encoded.
X_clean = X.copy()
for col in X_clean.columns:
    X_clean[col] = pd.to_numeric(X_clean[col], errors="ignore")
X_clean = pd.get_dummies(X_clean, drop_first=True)
X_clean = X_clean.replace([float("inf"), float("-inf")], pd.NA)
X_clean = X_clean.fillna(X_clean.median(numeric_only=True)).fillna(0)

print("Prepared feature shape:", X_clean.shape)
print("Class distribution:")
print(y_labels.value_counts())

## 3. Split into training and testing data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_labels,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_labels
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 4. Train machine-learning models

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=12),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
    ]),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE)
}

results = []
trained_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_models[model_name] = model

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision (weighted)": precision_score(y_test, predictions, average="weighted", zero_division=0),
        "Recall (weighted)": recall_score(y_test, predictions, average="weighted", zero_division=0),
        "F1-score (weighted)": f1_score(y_test, predictions, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score (weighted)", ascending=False)
display(results_df.round(4))

## 5. Detailed evaluation and confusion matrices

In [ ]:
class_order = sorted(y_labels.unique())

for model_name, model in trained_models.items():
    predictions = model.predict(X_test)
    print(f"\\n===== {model_name} =====")
    print(classification_report(y_test, predictions, zero_division=0))
    cm = confusion_matrix(y_test, predictions, labels=class_order)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order)
    display.plot()
    plt.title(f"Confusion Matrix — {model_name}")
    plt.tight_layout()
    plt.show()

## 6. Compare model performance

In [ ]:
metrics_to_plot = ["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-score (weighted)"]
results_df.set_index("Model")[metrics_to_plot].plot(kind="bar", ylim=(0, 1), figsize=(10, 5))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 7. Predict using a sample row

This example predicts the class of one held-out record. It does not accept a raw URL; the input must contain the same extracted features and columns used during training.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

sample_features = X_test.iloc[[0]]
sample_prediction = best_model.predict(sample_features)[0]

print("Selected model:", best_model_name)
print("Predicted class:", sample_prediction)
print("Actual class:", y_test.iloc[0])

## 8. Conclusion

The notebook compares three supervised-learning algorithms for phishing website classification. Use the evaluation table and confusion matrices generated above to identify the best-performing model on this test split. Do not write performance values into the final report until the notebook has been run successfully.

## Limitations and responsible use
- The dataset is historical and may not represent current phishing techniques.
- Classification errors are possible; a legitimate site can be flagged and a phishing site can be missed.
- The model uses pre-extracted features, not a live browser inspection.
- Never enter passwords, payment details, or personal information into a suspicious website.
- Cite the UCI dataset and follow its CC BY 4.0 license terms.